# OCR-оценка качества выпрямления (Tesseract)

## Setup environment

### Colab

In [ ]:
!git clone https://github.com/trxxnk/text-image-alignment.git

In [ ]:
import os
os.chdir("/content/text-image-alignment/")
print(f"Working directory: {os.getcwd()}")

In [ ]:
!git checkout dev

In [ ]:
!chmod +x src/scripts/setup_colab.sh
!src/scripts/setup_colab.sh

### OCR-зависимости (Colab)

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-rus tesseract-ocr-eng
!pip -q install pytesseract
!tesseract --version | head -1

### Local

In [ ]:
import os, sys
os.chdir(os.path.dirname(sys.prefix))
print(f"Working directory: {os.getcwd()}")

## Import libs

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import random

import cv2
import numpy as np
import torch
from torchvision.transforms import v2
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import pytesseract

In [ ]:
from src.tps_dewarp.dataset import TPSDataset, CanvasSpatialSpec
from src.tps_dewarp.geometry import build_remap_from_delta_tps
from src.tps_dewarp.training import load_train_config, build_model

## Config

In [ ]:
DATASET_DIR = "data/generated/v3"
CANVAS_MODEL = 256      # вход модели (как при обучении)
OCR_SIZE = 1024         # размер канвы для выпрямления + OCR (текст должен быть читаем)
GRID_SIZE = 9           # 9x9 = 81 контрольная точка
OCR_LANG = "rus+eng"    # языки Tesseract
N_EVAL = 40             # сколько примеров взять для метрики (OCR медленный)
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## CER / WER (без внешних зависимостей)

In [ ]:
def _levenshtein(a, b) -> int:
    """Расстояние редактирования для двух последовательностей (строк или списков слов)."""
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i - 1]
        for j in range(1, lb + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[lb]


def _norm(text: str) -> str:
    return " ".join(text.split())


def cer(ref: str, hyp: str) -> float:
    ref, hyp = _norm(ref), _norm(hyp)
    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0
    return _levenshtein(list(ref), list(hyp)) / len(ref)


def wer(ref: str, hyp: str) -> float:
    r, h = _norm(ref).split(), _norm(hyp).split()
    if len(r) == 0:
        return 0.0 if len(h) == 0 else 1.0
    return _levenshtein(r, h) / len(r)

## OCR + dewarp helpers

In [ ]:
def ocr_text(img_u8: np.ndarray, lang: str = OCR_LANG, psm: int = 6) -> str:
    return pytesseract.image_to_string(img_u8, lang=lang, config=f"--oem 1 --psm {psm}")


def to_uint8(img_t: torch.Tensor, normalized: bool = False) -> np.ndarray:
    """(1,H,W) -> uint8 HW. normalized=True для входа после Normalize(0.5,0.5)."""
    x = img_t.squeeze(0)
    if normalized:
        x = x * 0.5 + 0.5
    return (x * 255).clamp(0, 255).to(torch.uint8).cpu().numpy()


def dewarp(warped_u8: np.ndarray, delta, grid_size: int = GRID_SIZE) -> np.ndarray:
    """Выпрямление: delta в нормализованных координатах канвы (форма (81,2) или (162,))."""
    delta = np.asarray(delta, dtype=np.float64).reshape(grid_size * grid_size, 2)
    h, w = warped_u8.shape
    map_x, map_y = build_remap_from_delta_tps(delta, H=h, W=w, grid_size=grid_size, clip=False)
    return cv2.remap(
        warped_u8, map_x, map_y,
        interpolation=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )


@torch.no_grad()
def predict_delta(model, img_model_t: torch.Tensor, grid_size: int = GRID_SIZE) -> np.ndarray:
    out = model(img_model_t.unsqueeze(0).to(device)).cpu().numpy()
    return out.reshape(grid_size * grid_size, 2)

## Dataset

In [ ]:
spec_model = CanvasSpatialSpec(CANVAS_MODEL, CANVAS_MODEL, mode="letterbox", fill=0.0)
spec_hi = CanvasSpatialSpec(OCR_SIZE, OCR_SIZE, mode="letterbox", fill=1.0)
photometric = v2.Normalize(mean=[0.5], std=[0.5])

# Вход модели: 256, нормализованный
ds_model = TPSDataset(
    DATASET_DIR, spatial_spec=spec_model, photometric_transform=photometric,
    return_meta=True, lru_cache_maxsize=15,
)
# Hi-res для выпрямления и OCR: без фотометрии, фон белый
ds_hi = TPSDataset(
    DATASET_DIR, spatial_spec=spec_hi, photometric_transform=None,
    return_meta=True, lru_cache_maxsize=15,
)
len(ds_model), len(ds_hi)

## Загрузка модели

In [ ]:
# Архитектуру берём из того же YAML, что использовался при обучении
MODEL_CONFIG = "configs/train_unet.yaml"        # или configs/train_resunet.yaml
CHECKPOINT = "models/checkpoints/best.pt"        # или final.pt / путь с Google Drive

cfg = load_train_config(MODEL_CONFIG)
model = build_model(cfg).to(device)
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("loaded:", cfg.model_name, "| epoch:", ckpt.get("epoch"))

# --- Альтернатива: из MLflow Model Registry ---
# import mlflow.pytorch
# model = mlflow.pytorch.load_model("models:/<name>/<version>", map_location=device).to(device).eval()

## Качественный пример (один документ)

In [ ]:
i = 3337  # любой индекс не-identity примера
img_m, _, diff, meta = ds_model[i]
img_hi, delta_gt, _, _ = ds_hi[i]

warped_u8 = to_uint8(img_hi)
ref_u8 = cv2.imread(str(Path(DATASET_DIR) / meta["original"]), cv2.IMREAD_GRAYSCALE)
gt_u8 = dewarp(warped_u8, delta_gt.numpy())
model_u8 = dewarp(warped_u8, predict_delta(model, img_m))

ref_txt = ocr_text(ref_u8)
panels = {"original (ref)": ref_u8, "warped": warped_u8, "gt_dewarp": gt_u8, "model_dewarp": model_u8}

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
for ax, (name, im) in zip(axes, panels.items()):
    ax.imshow(im, cmap="gray"); ax.axis("off")
    if name == "original (ref)":
        ax.set_title(f"{name}\n(difficulty={diff})")
    else:
        h = ocr_text(im)
        ax.set_title(f"{name}\nCER={cer(ref_txt, h):.3f}  WER={wer(ref_txt, h):.3f}")
plt.tight_layout(); plt.show()

## Агрегированная метрика по выборке

In [ ]:
random.seed(SEED)

pool = [i for i in range(len(ds_hi)) if not ds_hi.samples[i]["is_identity"]]
random.shuffle(pool)
sample_idx = pool[:N_EVAL]

VARIANTS = ["warped", "gt_dewarp", "model_dewarp"]
agg = {v: {"cer": [], "wer": []} for v in VARIANTS}
by_diff = defaultdict(lambda: {v: {"cer": [], "wer": []} for v in VARIANTS})

for i in tqdm(sample_idx):
    img_m, _, diff, meta = ds_model[i]
    img_hi, delta_gt, _, _ = ds_hi[i]
    ref_u8 = cv2.imread(str(Path(DATASET_DIR) / meta["original"]), cv2.IMREAD_GRAYSCALE)
    if ref_u8 is None:
        continue
    warped_u8 = to_uint8(img_hi)
    ref_txt = ocr_text(ref_u8)
    variants = {
        "warped": warped_u8,
        "gt_dewarp": dewarp(warped_u8, delta_gt.numpy()),
        "model_dewarp": dewarp(warped_u8, predict_delta(model, img_m)),
    }
    for name, im in variants.items():
        h = ocr_text(im)
        c, w = cer(ref_txt, h), wer(ref_txt, h)
        agg[name]["cer"].append(c); agg[name]["wer"].append(w)
        by_diff[diff][name]["cer"].append(c); by_diff[diff][name]["wer"].append(w)

print(f"\n=== Среднее по выборке (n={len(sample_idx)}) ===")
print(f"{'variant':14s} {'CER':>8s} {'WER':>8s}")
for v in VARIANTS:
    print(f"{v:14s} {np.mean(agg[v]['cer']):8.3f} {np.mean(agg[v]['wer']):8.3f}")

In [ ]:
print("=== По уровням сложности (CER / WER) ===")
for diff in sorted(by_diff):
    n = len(by_diff[diff]["warped"]["cer"])
    print(f"\n[{diff}] n={n}")
    for v in VARIANTS:
        print(f"  {v:14s} CER={np.mean(by_diff[diff][v]['cer']):.3f}  WER={np.mean(by_diff[diff][v]['wer']):.3f}")

In [ ]:
results = {
    "n": len(sample_idx),
    "model": cfg.model_name,
    "overall": {v: {"cer": float(np.mean(agg[v]["cer"])), "wer": float(np.mean(agg[v]["wer"]))} for v in VARIANTS},
    "by_difficulty": {
        d: {v: {"cer": float(np.mean(by_diff[d][v]["cer"])), "wer": float(np.mean(by_diff[d][v]["wer"]))} for v in VARIANTS}
        for d in by_diff
    },
}
with open("ocr_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(json.dumps(results, ensure_ascii=False, indent=2))